# Data Cleaning & Formatting

**Capstone · Late Delivery Risk Prediction**  
**Data Lead** — Supply chain e-commerce dataset.

This notebook documents every step of data cleaning in order. The output (cleaned DataFrame `work`) is the handoff for EDA and modeling in the main pipeline.

---

## 1. Setup and load raw data

### 1.1 Imports and settings

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 50)
RNG = 42

### 1.2 Load raw data

Load the CSV from the project data folder (or Colab path). Use `low_memory=False` and `encoding='latin1'` to avoid mixed-type and encoding issues.

In [ ]:
candidates = [
    Path('../data/supply_chain.csv'), Path('data/supply_chain.csv'), Path('supply_chain.csv'),
    Path('../data/Data.csv'), Path('data/Data.csv'), Path('Data.csv'),
]
data_path = None
for p in candidates:
    if p.exists():
        data_path = p
        break
if data_path is None:
    for folder in [Path('../data'), Path('data'), Path('.')]:
        if folder.exists():
            csvs = list(folder.glob('*.csv'))
            if csvs:
                data_path = csvs[0]
                break
if data_path is None:
    raise FileNotFoundError("No CSV found. Put dataset in data/ as supply_chain.csv or set data_path in Colab.")

df = pd.read_csv(data_path, low_memory=False, encoding='latin1')
print(f"Loaded: {data_path}")
df.shape

### 1.3 Initial shape and column overview

In [ ]:
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
print("\nColumn list:")
for i, c in enumerate(df.columns, 1):
    print(f"  {i:2}. {c}")

---

## 2. Data dictionary

Key columns, types, and notes for cleaning and modelling. Use with caution: some columns are dropped or excluded from features to avoid leakage.

| Column | Data type | Notes |
|--------|-----------|-------|
| **Targets** | | |
| Late_delivery_risk | int (0/1) | **Classification target.** Do not drop rows with missing value; drop those rows in cleaning. |
| Days for shipping (real) | int | Actual days to ship. Used to build Delay_Gap. |
| Days for shipment (scheduled) | int | Promised days. Used to build Delay_Gap. |
| **Categoricals (features)** | | |
| Customer Segment | str | Consumer, Corporate, Home Office. |
| Market | str | Market grouping. |
| Order Region | str | More granular than Market. |
| Order Country | str | High cardinality; may need grouping at modelling. |
| Order City | str | Very high cardinality — consider dropping at modelling. |
| Shipping Mode | str | Key feature (Standard Class, First Class, etc.). |
| Category Name | str | Product category. Prefer over Product Name. |
| Product Name | str | High cardinality; use Category Name instead at modelling. |
| Order Status | str | May contain post-delivery info; **use with caution** to avoid leakage. |
| Delivery Status | str | **Do not use as feature** — encodes outcome (leakage). |
| **Dates** | | |
| order date (DateOrders) | str → datetime | Parse in cleaning. |
| shipping date (DateOrders) | str → datetime | Parse in cleaning. |
| **IDs (often drop)** | | |
| Order Id | int | Primary key. One order, many rows (items). |
| Order Item Id | int | One order can have multiple items. |
| **PII — drop** | | |
| Customer Fname, Lname, Email, Password | str | No predictive use; privacy. **Drop.** |
| Customer Street, City, Country, State, Zipcode | str | **Drop.** |
| **Other — drop** | | |
| Order Zipcode, Product Description, Product Image | — | Sparse/irrelevant. **Drop.** |
| Order Customer Id, Order Item Cardprod Id, Product Category Id | — | Redundant with names. **Drop.** |

---

## 3. Data cleaning & formatting

Steps 3.1–3.8 are applied in order. Final output: DataFrame `work`.

### 3.1 Drop columns flagged for removal

- **PII:** Customer Fname, Lname, Email, Password, Street, City, Country, State, Zipcode (no predictive value, privacy).
- **Sparse/irrelevant:** Order Zipcode, Product Description, Product Image, Order Profit Per Order.
- **Redundant IDs:** Order Customer Id, Order Item Cardprod Id, Product Category Id (we keep Category Name, not ID).

In [ ]:
COLS_PII = [
    'Customer Fname', 'Customer Lname', 'Customer Email', 'Customer Password',
    'Customer Street', 'Customer City', 'Customer Country', 'Customer State', 'Customer Zipcode'
]
COLS_DROP = ['Order Zipcode', 'Product Description', 'Product Image', 'Order Profit Per Order']
COLS_REDUNDANT_ID = ['Order Customer Id', 'Order Item Cardprod Id', 'Product Category Id']

to_drop = [c for c in COLS_PII + COLS_DROP + COLS_REDUNDANT_ID if c in df.columns]
work = df.drop(columns=to_drop, errors='ignore')

print(f"Dropped {len(to_drop)} columns: {to_drop}")
print(f"Shape after 3.1: {work.shape[0]:,} rows × {work.shape[1]} columns")

### 3.2 Fix data types: parse dates

Order and shipping dates may be stored as strings or mixed formats. Convert to `datetime`; invalid values become `NaT` (`errors='coerce'`).

In [ ]:
work['order date (DateOrders)'] = pd.to_datetime(work['order date (DateOrders)'], errors='coerce')
work['shipping date (DateOrders)'] = pd.to_datetime(work['shipping date (DateOrders)'], errors='coerce')

print("Date columns dtypes after parsing:")
print(work[['order date (DateOrders)', 'shipping date (DateOrders)']].dtypes)
print(f"\nNaT count - order date: {work['order date (DateOrders)'].isna().sum()}, shipping date: {work['shipping date (DateOrders)'].isna().sum()}")

### 3.3 Missing value audit

Check missing counts per column. **Critical:** Rows with missing `Late_delivery_risk` cannot be used for training or evaluation — we drop them.

In [ ]:
TARGET = 'Late_delivery_risk'

missing = work.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print("Columns with missing values:")
    print(missing.to_string())
else:
    print("No missing values in any column (except dates if any NaT).")

target_missing = work[TARGET].isna().sum()
print(f"\nRows with missing {TARGET}: {target_missing}")
work = work.dropna(subset=[TARGET])
print(f"After dropping rows with missing target: {work.shape[0]:,} rows")

### 3.4 Duplicate check

Remove exact duplicate rows so each row represents a unique record for modelling.

In [ ]:
n_before = len(work)
n_dup = work.duplicated().sum()
work = work.drop_duplicates()
n_after = len(work)

print(f"Duplicate rows found: {n_dup}")
print(f"Rows before: {n_before:,}  |  after: {n_after:,}  |  removed: {n_before - n_after:,}")

### 3.5 Check for impossible / outlier values

Sanity check: days columns should be non-negative; optional checks on numeric ranges. We do not drop rows here unless values are clearly invalid (e.g. negative days).

In [ ]:
days_real = work['Days for shipping (real)']
days_sched = work['Days for shipment (scheduled)']

print("Days for shipping (real):", days_real.min(), "to", days_real.max(), "| mean", round(days_real.mean(), 2))
print("Days for shipment (scheduled):", days_sched.min(), "to", days_sched.max(), "| mean", round(days_sched.mean(), 2))

neg_real = (days_real < 0).sum()
neg_sched = (days_sched < 0).sum()
if neg_real > 0 or neg_sched > 0:
    print(f"\nWarning: negative values — real: {neg_real}, scheduled: {neg_sched}")
else:
    print("\nNo negative days; no rows removed in this step.")

### 3.6 Engineer the regression target (Delay_Gap)

**Delay_Gap** = Days for shipping (real) − Days for shipment (scheduled).  
- **Positive** = shipped later than promised (late by that many days).  
- **Zero or negative** = on time or early.  
This is the **regression target** for “how late?” in the main pipeline.

In [ ]:
TARGET_REG = 'Delay_Gap'
work[TARGET_REG] = work['Days for shipping (real)'] - work['Days for shipment (scheduled)']

print(f"{TARGET_REG} stats: min={work[TARGET_REG].min()}, max={work[TARGET_REG].max()}, mean={work[TARGET_REG].mean():.2f}")
work[[TARGET_REG]].describe()

### 3.7 Extract date features (optional)

Optional: derive year, month, day-of-week from order date for later use in EDA or modelling. Kept minimal here; can be extended.

In [ ]:
work['order_year'] = work['order date (DateOrders)'].dt.year
work['order_month'] = work['order date (DateOrders)'].dt.month
work['order_dow'] = work['order date (DateOrders)'].dt.dayofweek  # 0=Monday

print("Date features added: order_year, order_month, order_dow")
work[['order_year', 'order_month', 'order_dow']].head()

### 3.8 Final cleaned dataset overview

Summary of the cleaned DataFrame `work`: shape, dtypes, and a few validation checks. This is the **handoff** for the rest of the pipeline.

In [ ]:
print("=" * 60)
print("FINAL CLEANED DATASET (work)")
print("=" * 60)
print(f"Shape: {work.shape[0]:,} rows × {work.shape[1]} columns")
print(f"\nDuplicates: {work.duplicated().sum()}")
print(f"Missing {TARGET}: {work[TARGET].isna().sum()}")
print(f"\nTarget distribution:")
print(work[TARGET].value_counts().to_string())
print(f"\nLate rate: {work[TARGET].mean():.2%}")
print("\nDtypes:")
work.info()

In [ ]:
work.head(10)

---

**End of data cleaning.**  
Save `work` (e.g. to CSV or use in the main notebook) for EDA and modelling.